In [24]:
import random

def gerar_teste_grande(nome_arquivo, num_acoes=5000, num_operacoes=2000):
    with open(nome_arquivo, 'w') as f:
        # Configuração inicial: M=3, RET, STAB, AVGRET
        f.write(f"M 3 RET STAB AVGRET\n")
        
        # Registra as ações
        for i in range(num_acoes):
            f.write(f"A {i}\n")
            
        # Cadastra um cliente
        f.write("U 0\n")
        
        # Coloca o cliente com muitas ações na carteira
        for i in range(num_acoes // 2):
            f.write(f"B 0 {i}\n")
            
        # Gera cotações (P) e Consultas (Q)
        for i in range(num_operacoes):
            acao_id = random.randint(0, num_acoes - 1)
            preco = round(random.uniform(10.0, 100.0), 2)
            f.write(f"P {acao_id} {preco}\n")
            
            # A cada 500 cotações, faz uma consulta pesada
            if i % 500 == 0:
                f.write(f"Q {i} 0 10 2 RET 1.0 STAB 1.0\n")

print("Arquivo muitostestes.txt gerado com sucesso!")
gerar_teste_grande("entradas/in_muitostestes.txt")

Arquivo muitostestes.txt gerado com sucesso!


In [29]:
import pandas as pd
import subprocess
import os

# Configurações
EXECUTAVEL = "./bin/tp1.out"
DIRETORIO_ENTRADAS = "entradas"

def rodar_testes():
    resultados = []
    arquivos = sorted([f for f in os.listdir(DIRETORIO_ENTRADAS) if f.endswith(".txt")])
    
    for arq in arquivos:
        caminho = os.path.join(DIRETORIO_ENTRADAS, arq)
        # O C++ está fixo como Sob Demanda internamente
        comando = [EXECUTAVEL] 
        
        with open(caminho, 'r') as f:
            proc = subprocess.run(comando, stdin=f, capture_output=True, text=True)
            
        # Procura a linha de estatísticas na saída do seu C++
        for linha in proc.stdout.split('\n'):
            if linha.startswith("ESTATISTICAS"):
                partes = linha.split(',')
                resultados.append({
                    "Arquivo": arq,
                    "Estrategia": "Sob Demanda",
                    "N": int(partes[2]),
                    "Tempo Consultas (ms)": float(partes[5]),
                    "Reconstrucoes": int(partes[6])
                })
    
    df = pd.DataFrame(resultados)
    # Salva para usarmos depois na comparação final
    df.to_csv("resultados_sob_demanda.csv", index=False)
    return df

df_sob_demanda = rodar_testes()
print("Dados da Sob Demanda coletados e salvos!")
display(df_sob_demanda)

Dados da Sob Demanda coletados e salvos!


,Arquivo,Estrategia,N,Tempo Consultas (ms),Reconstrucoes
0,in_0.txt,Sob Demanda,6,0.07,0
1,in_1.txt,Sob Demanda,1,0.05,0
2,in_2.txt,Sob Demanda,2,0.05,0
3,in_3.txt,Sob Demanda,2,0.08,0
4,in_4.txt,Sob Demanda,2,0.06,0
5,in_5.txt,Sob Demanda,3,0.03,0
6,in_6.txt,Sob Demanda,3,0.03,0
7,in_7.txt,Sob Demanda,3,0.05,0
8,in_8.txt,Sob Demanda,2,0.03,0
9,in_9.txt,Sob Demanda,2,0.03,0


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Ordena por N para a linha fazer sentido
df_plot = df_sob_demanda.sort_values(by="N")

plt.plot(df_plot["N"], df_plot["Tempo Consultas (ms)"], 
         marker='s', color='#2980b9', label='Sob Demanda (Custo no Q)')

plt.yscale('log') # Fundamental para ver os microssegundos
plt.title("Análise de Desempenho: Estratégia Sob Demanda", fontsize=14)
plt.xlabel("Número de Ações (N)", fontsize=12)
plt.ylabel("Tempo de Consulta (ms) - Escala Log", fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()